In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1e9,
            2
        ),
        "GB"
    )

PyTorch: 2.14.1+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB
GPU memory: 17.06 GB


In [2]:
import pandas as pd

claims_df = pd.read_csv("fenice_validation_alignscore.csv")

print("Total extracted claims:", len(claims_df))
print(
    "Total articles:",
    claims_df.groupby(["event_id", "article_id"]).ngroups
)
print("Total events:", claims_df["event_id"].nunique())

display(claims_df.head())

Total extracted claims: 1014
Total articles: 30
Total events: 6


,event_id,article_id,passage_id,claim_id,claim,source_passage,alignscore,support_flag
0,6,6.1,6.1_p001,1,Fruit Sold In CA Recalled Due To Hepatitis A O...,Fruit Sold In CA Recalled Due To Hepatitis A O...,0.984695,True
1,6,6.1,6.1_p001,2,Different brands of frozen organic strawberrie...,Fruit Sold In CA Recalled Due To Hepatitis A O...,0.986042,True
2,6,6.1,6.1_p001,3,The FDA recalled two frozen strawberry and tro...,Fruit Sold In CA Recalled Due To Hepatitis A O...,0.996483,True
3,6,6.1,6.1_p001,4,"Scenic Fruit Company of Gresham, Oregon recall...",Fruit Sold In CA Recalled Due To Hepatitis A O...,0.987993,True
4,6,6.1,6.1_p001,5,"Scenic Fruit Company of Gresham, Oregon recall...",Fruit Sold In CA Recalled Due To Hepatitis A O...,0.978279,True


In [3]:
# Selecting the first article in the dataset
article_keys = (
    claims_df[["event_id", "article_id"]]
    .drop_duplicates()
)

event_id, article_id = article_keys.iloc[0]

article_df = claims_df[
    (claims_df["event_id"] == event_id) &
    (claims_df["article_id"] == article_id)
].copy().reset_index(drop=True)

article_claims = article_df["claim"].tolist()

print("Event:", event_id)
print("Article:", article_id)
print("Number of claims:", len(article_claims))

for i, claim in enumerate(article_claims, 1):
    print(f"{i}. {claim}")

Event: 6.0
Article: 6.1
Number of claims: 22
1. Fruit Sold In CA Recalled Due To Hepatitis A Outbreak Frozen organic strawberries and fruit sold throughout California were linked to a possible outbreak of Hepatitis A.
2. Different brands of frozen organic strawberries sold in California were linked to a recent Hepatitis A outbreak.
3. The FDA recalled two frozen strawberry and tropical fruit products sold in California recently.
4. Scenic Fruit Company of Gresham, Oregon recalled Simply Nature frozen organic strawberries sold in Aldi stores.
5. Scenic Fruit Company of Gresham, Oregon recalled an organic tropical fruit blend sold in Trader Joe's grocery stores nationwide due to the outbreak.
6. California Splendor of San Diego recalled bags of their Kirkland Signature Frozen Organic Strawberries.
7. The bags were sold at Costco stores in Los Angeles, two San Diego Business Centers and Hawaii.
8. The company has ceased the production and distribution of the product.
9. The FDA and the co

In [4]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("GPU memory (GB):",
      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

GPU: Tesla P100-PCIE-16GB
Compute capability: (6, 0)
GPU memory (GB): 17.06


In [5]:
from llama_cpp import Llama
print("llama-cpp-python imported successfully!")

llama-cpp-python imported successfully!


In [6]:
import sys
import subprocess

# Install Hugging Face Hub if needed
subprocess.check_call([
    sys.executable, "-m", "pip",
    "install", "-q", "huggingface_hub"
])

from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="Qwen/Qwen3-8B-GGUF",
    filename="Qwen3-8B-Q4_K_M.gguf",
    local_dir="/project2/xiangren_1987/group_22/models"
)

print("Model downloaded successfully!")
print("Path:", model_path)

Model downloaded successfully!
Path: /project2/xiangren_1987/group_22/models/Qwen3-8B-Q4_K_M.gguf


In [7]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    n_threads=4,
    verbose=True
)
print("Qwen3-8B loaded successfully!")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 16269 MiB):
  Device 0: Tesla P100-PCIE-16GB, compute capability 6.0, VMM: yes, VRAM: 16269 MiB
llama_model_loader: loaded meta data with 28 key-value pairs and 399 tensors from /project2/xiangren_1987/group_22/models/Qwen3-8B-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen3 8B Awq Compatible Instruct
llama_model_loader: - kv   3:                           general.finetune str              = awq-compatible-Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen3
llama_model_loader: - kv   5:                      

Qwen3-8B loaded successfully!


Using gguf chat template: {%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}
{%- for index in range(ns.last_query_index, -1, -1

In [8]:
REDUNDANCY_PROMPT = """
You are evaluating semantic redundancy in factual claims
extracted from news articles.

Your task is to identify claims that repeat substantially
the same factual information.

Two claims are redundant if:
- They express the same core factual assertion.
- Differences in wording do not change their meaning.
- Keeping both provides little or no additional
  factual information.

Two claims are NOT redundant if:
- They describe different events or actions.
- They differ meaningfully in entities, numbers,
  dates, negation, or uncertainty.
- One provides substantial additional factual
  information absent from the other.
- They merely discuss the same general topic.

Instructions:
- Evaluate meaning, not word overlap.
- Do not assume duplicates exist.
- Do not rewrite claims.
- Each claim ID can appear in at most one group.
- Return only groups with at least two claims.

Return valid JSON:
{"duplicate_groups": []}
"""

In [9]:
import json

numbered_claims = "\n".join(
    f"{i}. {claim}"
    for i, claim in enumerate(article_claims, 1)
)
response = llm.create_chat_completion(
    messages=[
        {
            "role": "system",
            "content": REDUNDANCY_PROMPT
        },
        {
            "role": "user",
            "content": (
                "/no_think\n"
                "CLAIMS TO EVALUATE:\n"
                + numbered_claims
            )
        }
    ],
    temperature=0,
    max_tokens=300,
    response_format={
        "type": "json_object"
    }
)

generated = response["choices"][0]["message"]["content"]

print("RAW RESPONSE:")
print(generated)

result = json.loads(generated)
duplicate_groups = result["duplicate_groups"]
print("\nDuplicate groups:", duplicate_groups)

ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =    1550.85 ms /   870 tokens (    1.78 ms per token,   560.98 tokens per second)
llama_perf_context_print:        eval time =    3016.22 ms /    98 runs   (   30.78 ms per token,    32.49 tokens per second)
llama_perf_context_print:       total time =    8266.39 ms /   968 tokens
llama_perf_context_print:    graphs reused =         97


RAW RESPONSE:
{"duplicate_groups": [{"group": ["1", "2"]}, {"group": ["3", "17"]}, {"group": ["4", "6", "17"]}, {"group": ["5", "17"]}, {"group": ["11", "15"]}, {"group": ["14", "17"]}, {"group": ["16", "17"]}, {"group": ["21", "22"]}]}


Duplicate groups: [{'group': ['1', '2']}, {'group': ['3', '17']}, {'group': ['4', '6', '17']}, {'group': ['5', '17']}, {'group': ['11', '15']}, {'group': ['14', '17']}, {'group': ['16', '17']}, {'group': ['21', '22']}]


In [10]:
import pandas as pd

redundant_copies = sum(
    len(group) - 1
    for group in duplicate_groups
)

redundancy_rate = (
    redundant_copies / len(article_claims)
)

print("Total claims:", len(article_claims))
print("Duplicate groups:", duplicate_groups)
print("Redundant copies:", redundant_copies)
print(f"Redundancy rate: {redundancy_rate:.2%}")

results_df = pd.DataFrame({
    "event_id": [event_id],
    "article_id": [article_id],
    "total_claims": [len(article_claims)],
    "duplicate_groups": [json.dumps(duplicate_groups)],
    "redundant_copies": [redundant_copies],
    "redundancy_rate": [redundancy_rate]
})

results_df.to_csv(
    "qwen_redundancy_one_article.csv",
    index=False
)

print("Results saved!")

Total claims: 22
Duplicate groups: [{'group': ['1', '2']}, {'group': ['3', '17']}, {'group': ['4', '6', '17']}, {'group': ['5', '17']}, {'group': ['11', '15']}, {'group': ['14', '17']}, {'group': ['16', '17']}, {'group': ['21', '22']}]
Redundant copies: 0
Redundancy rate: 0.00%
Results saved!


In [11]:
import huggingface_hub
import transformers

print("Hugging Face Hub:", huggingface_hub.__version__)
print("Transformers:", transformers.__version__)

from sentence_transformers import SentenceTransformer

print("Sentence Transformers imported successfully!")

Hugging Face Hub: 1.33.0
Transformers: 5.20.1
Sentence Transformers imported successfully!


In [13]:
from sentence_transformers import SentenceTransformer
import torch

sbert = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device="cpu"
)

print("SBERT loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT loaded successfully!


In [14]:
import numpy as np
import pandas as pd
from sentence_transformers import util

# article_claims contains the 22 FENICE claims
embeddings = sbert.encode(
    article_claims,
    convert_to_tensor=True,
    normalize_embeddings=True
)

similarity_matrix = util.cos_sim(
    embeddings,
    embeddings
).cpu().numpy()

pairs = []

for i in range(len(article_claims)):
    for j in range(i + 1, len(article_claims)):

        pairs.append({
            "Claim 1 ID": i + 1,
            "Claim 2 ID": j + 1,
            "Similarity": float(similarity_matrix[i, j]),
            "Claim 1": article_claims[i],
            "Claim 2": article_claims[j]
        })

pairs_df = pd.DataFrame(pairs)
pairs_df = pairs_df.sort_values(
    "Similarity",
    ascending=False
).reset_index(drop=True)

print("Total possible pairs:", len(pairs_df))
display(
    pairs_df.head(20)
)

Total possible pairs: 231


,Claim 1 ID,Claim 2 ID,Similarity,Claim 1,Claim 2
0,19,21,0.961154,The illness can cause liver failure in people ...,The FDA said that the illness can cause liver ...
1,20,22,0.959762,The illness can cause liver failure in people ...,The FDA said that the illness can cause liver ...
2,1,2,0.883355,Fruit Sold In CA Recalled Due To Hepatitis A O...,Different brands of frozen organic strawberrie...
3,2,15,0.868105,Different brands of frozen organic strawberrie...,An identical strain of hepatitis in May 2022 w...
4,1,17,0.859105,Fruit Sold In CA Recalled Due To Hepatitis A O...,Affected lots and products in California inclu...
5,21,22,0.854232,The FDA said that the illness can cause liver ...,The FDA said that the illness can cause liver ...
6,19,20,0.831124,The illness can cause liver failure in people ...,The illness can cause liver failure in people ...
7,19,22,0.820556,The illness can cause liver failure in people ...,The FDA said that the illness can cause liver ...
8,1,3,0.808464,Fruit Sold In CA Recalled Due To Hepatitis A O...,The FDA recalled two frozen strawberry and tro...
9,20,21,0.794553,The illness can cause liver failure in people ...,The FDA said that the illness can cause liver ...


In [15]:
for _, row in pairs_df.head(15).iterrows():

    print("=" * 65)
    print(
        f"Claims {int(row['Claim 1 ID'])} "
        f"& {int(row['Claim 2 ID'])}"
    )
    print(f"Similarity: {row['Similarity']:.4f}")

    print("\nClaim A:")
    print(row["Claim 1"])

    print("\nClaim B:")
    print(row["Claim 2"])
    print()

Claims 19 & 21
Similarity: 0.9612

Claim A:
The illness can cause liver failure in people who have severe pre-existing illnesses.

Claim B:
The FDA said that the illness can cause liver failure in people who have severe pre-existing illnesses.

Claims 20 & 22
Similarity: 0.9598

Claim A:
The illness can cause liver failure in people who are immunocompromised.

Claim B:
The FDA said that the illness can cause liver failure in people who are immunocompromised.

Claims 1 & 2
Similarity: 0.8834

Claim A:
Fruit Sold In CA Recalled Due To Hepatitis A Outbreak Frozen organic strawberries and fruit sold throughout California were linked to a possible outbreak of Hepatitis A.

Claim B:
Different brands of frozen organic strawberries sold in California were linked to a recent Hepatitis A outbreak.

Claims 2 & 15
Similarity: 0.8681

Claim A:
Different brands of frozen organic strawberries sold in California were linked to a recent Hepatitis A outbreak.

Claim B:
An identical strain of hepatitis i

In [16]:
PAIRWISE_PROMPT = """
You are evaluating semantic redundancy between two
factual claims extracted from a news article.

Determine whether the claims express substantially
the same factual information.

REDUNDANT:
Both claims communicate the same core factual assertion.
Keeping both adds little or no substantial information.

NOT_REDUNDANT:
The claims express distinct factual assertions, even
if they discuss the same topic or share many words.

Rules:
- Evaluate factual meaning, not word similarity.
- Different wording can express the same fact.
- Multiple entities or similar topics do not imply redundancy.
- Preserve meaningful differences in dates, numbers,
  negation, uncertainty, entities, and relationships.
- If one claim adds substantial new factual information,
  classify the pair as NOT_REDUNDANT.
- Do not assume the claims are redundant.
- Do not rewrite the claims.

Return ONLY valid JSON:
{
  "label": "REDUNDANT",
  "reason": "Brief explanation"
}
"""

In [17]:
import json

def evaluate_pair_qwen(claim_a, claim_b):

    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": PAIRWISE_PROMPT
            },
            {
                "role": "user",
                "content": (
                    "/no_think\n"
                    f"CLAIM A: {claim_a}\n"
                    f"CLAIM B: {claim_b}"
                )
            }
        ],
        temperature=0,
        max_tokens=160,
        response_format={"type": "json_object"}
    )

    generated = response["choices"][0]["message"]["content"]

    result = json.loads(generated)

    if result.get("label") not in {
        "REDUNDANT", "NOT_REDUNDANT"
    }:
        raise ValueError(f"Unexpected label: {result}")

    return result

print("Pairwise redundancy evaluator ready!")

Pairwise redundancy evaluator ready!


In [18]:
pair_results = []

for _, row in pairs_df.head(15).iterrows():

    result = evaluate_pair_qwen(
        row["Claim 1"],
        row["Claim 2"]
    )

    pair_results.append({
        "Claim 1 ID": int(row["Claim 1 ID"]),
        "Claim 2 ID": int(row["Claim 2 ID"]),
        "SBERT Similarity": row["Similarity"],
        "Qwen Label": result["label"],
        "Reason": result.get("reason", "")
    })

pair_results_df = pd.DataFrame(pair_results)

display(pair_results_df)

Llama.generate: 8 prefix-match hit, remaining 237 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =     483.18 ms /   237 tokens (    2.04 ms per token,   490.50 tokens per second)
llama_perf_context_print:        eval time =    1525.26 ms /    53 runs   (   28.78 ms per token,    34.75 tokens per second)
llama_perf_context_print:       total time =    4030.72 ms /   290 tokens
llama_perf_context_print:    graphs reused =         51
Llama.generate: 212 prefix-match hit, remaining 33 prompt tokens to eval
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =     230.00 ms /    33 tokens (    6.97 ms per token,   143.48 tokens per second)
llama_perf_context_print:        eval time =    1496.22 ms /    52 runs   (   28.77 ms per token,    34.75 tokens per second)
llama_perf_context_print:   

,Claim 1 ID,Claim 2 ID,SBERT Similarity,Qwen Label,Reason
0,19,21,0.961154,REDUNDANT,Both claims state the same factual assertion a...
1,20,22,0.959762,REDUNDANT,Both claims state the same factual assertion a...
2,1,2,0.883355,NOT_REDUNDANT,Claim A mentions the recall due to Hepatitis A...
3,2,15,0.868105,NOT_REDUNDANT,Claim A attributes the outbreak to different b...
4,1,17,0.859105,NOT_REDUNDANT,Claim A discusses a general recall of fruit in...
5,21,22,0.854232,NOT_REDUNDANT,The claims differ in the specific group affect...
6,19,20,0.831124,NOT_REDUNDANT,The claims differ in the specific condition de...
7,19,22,0.820556,NOT_REDUNDANT,Claim A refers to 'severe pre-existing illness...
8,1,3,0.808464,NOT_REDUNDANT,Claim A specifies the outbreak is due to Hepat...
9,20,21,0.794553,NOT_REDUNDANT,Claim A states the illness can cause liver fai...


In [19]:
THRESHOLD = 0.70

candidate_pairs = pairs_df[
    pairs_df["Similarity"] >= THRESHOLD
].copy()

print("Total possible pairs:", len(pairs_df))
print("Candidate pairs:", len(candidate_pairs))

display(
    candidate_pairs[
        ["Claim 1 ID", "Claim 2 ID", "Similarity"]
    ]
)

Total possible pairs: 231
Candidate pairs: 21


,Claim 1 ID,Claim 2 ID,Similarity
0,19,21,0.961154
1,20,22,0.959762
2,1,2,0.883355
3,2,15,0.868105
4,1,17,0.859105
5,21,22,0.854232
6,19,20,0.831124
7,19,22,0.820556
8,1,3,0.808464
9,20,21,0.794553


In [20]:
# Find pairs we already evaluated
evaluated_pairs = set(
    zip(
        pair_results_df["Claim 1 ID"],
        pair_results_df["Claim 2 ID"]
    )
)

# Keep only pairs not yet evaluated
remaining_pairs = candidate_pairs[
    ~candidate_pairs.apply(
        lambda row: (
            row["Claim 1 ID"],
            row["Claim 2 ID"]
        ) in evaluated_pairs,
        axis=1
    )
]

print("Remaining pairs:", len(remaining_pairs))

new_results = []

for _, row in remaining_pairs.iterrows():

    result = evaluate_pair_qwen(
        row["Claim 1"],
        row["Claim 2"]
    )

    new_results.append({
        "Claim 1 ID": int(row["Claim 1 ID"]),
        "Claim 2 ID": int(row["Claim 2 ID"]),
        "SBERT Similarity": row["Similarity"],
        "Qwen Label": result["label"],
        "Reason": result.get("reason", "")
    })

new_results_df = pd.DataFrame(new_results)

display(new_results_df)

Llama.generate: 203 prefix-match hit, remaining 244 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture


Remaining pairs: 6


llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =     488.97 ms /   244 tokens (    2.00 ms per token,   499.01 tokens per second)
llama_perf_context_print:        eval time =    2305.72 ms /    79 runs   (   29.19 ms per token,    34.26 tokens per second)
llama_perf_context_print:       total time =    5833.82 ms /   323 tokens
llama_perf_context_print:    graphs reused =         77
Llama.generate: 203 prefix-match hit, remaining 45 prompt tokens to eval
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =     233.90 ms /    45 tokens (    5.20 ms per token,   192.39 tokens per second)
llama_perf_context_print:        eval time =    2081.97 ms /    72 runs   (   28.92 ms per token,    34.58 tokens per second)
llama_perf_context_print:       total time =    5071.08 ms /   117 tokens
llama_perf_context_print:    graphs reused =         70
Llama.generate: 203 prefix-match hit, remaini

,Claim 1 ID,Claim 2 ID,SBERT Similarity,Qwen Label,Reason
0,4,17,0.748337,NOT_REDUNDANT,Claim A focuses on a specific recall of Simply...
1,3,4,0.747503,NOT_REDUNDANT,Claim A mentions a recall of two frozen strawb...
2,4,5,0.747274,NOT_REDUNDANT,The claims mention different products (strawbe...
3,6,17,0.747105,NOT_REDUNDANT,Claim A mentions a specific product (Kirkland ...
4,3,17,0.745597,NOT_REDUNDANT,Claim A mentions a recall of two frozen fruit ...
5,1,6,0.707312,NOT_REDUNDANT,Claim A discusses a broader recall involving m...


In [21]:
all_pair_results_df = pd.concat(
    [pair_results_df, new_results_df],
    ignore_index=True
)

print("Total evaluated pairs:", len(all_pair_results_df))

print("\nQwen label counts:")
print(all_pair_results_df["Qwen Label"].value_counts())

print("\nRedundant pairs:")
display(
    all_pair_results_df[
        all_pair_results_df["Qwen Label"] == "REDUNDANT"
    ]
)

Total evaluated pairs: 21

Qwen label counts:
Qwen Label
NOT_REDUNDANT    19
REDUNDANT         2
Name: count, dtype: int64

Redundant pairs:


,Claim 1 ID,Claim 2 ID,SBERT Similarity,Qwen Label,Reason
0,19,21,0.961154,REDUNDANT,Both claims state the same factual assertion a...
1,20,22,0.959762,REDUNDANT,Both claims state the same factual assertion a...


In [22]:
all_pair_results_df.to_csv(
    "qwen_redundancy_pairs_article_6_1.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


**Since it ran for 1 article, now running it for all 30 articles in the validation set**

In [23]:
import pandas as pd
import numpy as np
import json
import time
from pathlib import Path
from sentence_transformers import util

# Check that existing objects are available
print("Claims loaded:", "claims_df" in globals())
print("SBERT loaded:", "sbert" in globals())
print("Qwen loaded:", "llm" in globals())
print("Pairwise evaluator ready:",
      "evaluate_pair_qwen" in globals())

print("Total claims:", len(claims_df))
print("Total articles:",
      claims_df.groupby(["event_id", "article_id"]).ngroups)

Claims loaded: True
SBERT loaded: True
Qwen loaded: True
Pairwise evaluator ready: True
Total claims: 1014
Total articles: 30


In [24]:
RESULTS_DIR = Path(
    "/project2/xiangren_1987/group_22/redundancy_results"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD = 0.70

ARTICLE_FILE = RESULTS_DIR / "redundancy_by_article.csv"
PAIRS_FILE = RESULTS_DIR / "redundancy_pairs.csv"
SUMMARY_FILE = RESULTS_DIR / "redundancy_summary.csv"

print("Results directory:", RESULTS_DIR)
print("SBERT threshold:", THRESHOLD)

Results directory: /project2/xiangren_1987/group_22/redundancy_results
SBERT threshold: 0.7


In [25]:
def evaluate_article(event_id, article_id, group_df):

    claims = group_df["claim"].tolist()
    n = len(claims)

    # Generate SBERT embeddings
    embeddings = sbert.encode(
        claims,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    similarity_matrix = util.cos_sim(
        embeddings, embeddings
    ).cpu().numpy()

    pair_records = []

    # Evaluate only candidate pairs above threshold
    for i in range(n):
        for j in range(i + 1, n):

            similarity = float(similarity_matrix[i, j])

            if similarity < THRESHOLD:
                continue

            result = evaluate_pair_qwen(
                claims[i],
                claims[j]
            )

            pair_records.append({
                "event_id": event_id,
                "article_id": article_id,
                "claim_1_id": i + 1,
                "claim_2_id": j + 1,
                "claim_1": claims[i],
                "claim_2": claims[j],
                "sbert_similarity": similarity,
                "qwen_label": result["label"],
                "reason": result.get("reason", "")
            })

    # Build connected components of redundant claims
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for record in pair_records:
        if record["qwen_label"] == "REDUNDANT":
            union(
                record["claim_1_id"] - 1,
                record["claim_2_id"] - 1
            )

    clusters = {}

    for i in range(n):
        root = find(i)
        clusters.setdefault(root, []).append(i + 1)

    duplicate_groups = [
        members
        for members in clusters.values()
        if len(members) > 1
    ]

    redundant_copies = sum(
        len(group) - 1
        for group in duplicate_groups
    )

    article_result = {
        "event_id": event_id,
        "article_id": article_id,
        "total_claims": n,
        "candidate_pairs": len(pair_records),
        "redundant_pairs": sum(
            r["qwen_label"] == "REDUNDANT"
            for r in pair_records
        ),
        "duplicate_groups": json.dumps(duplicate_groups),
        "redundant_copies": redundant_copies,
        "redundancy_rate": redundant_copies / n if n else 0,
        "status": "OK"
    }

    return article_result, pair_records

print("Article evaluation function ready!")

Article evaluation function ready!


In [26]:
# Load previous progress if available
if ARTICLE_FILE.exists():
    article_results_df = pd.read_csv(ARTICLE_FILE)
    print("Loaded previous article results.")
else:
    article_results_df = pd.DataFrame()

if PAIRS_FILE.exists():
    all_pairs_df = pd.read_csv(PAIRS_FILE)
    print("Loaded previous pair results.")
else:
    all_pairs_df = pd.DataFrame()

grouped = claims_df.groupby(
    ["event_id", "article_id"],
    sort=False
)

total_articles = grouped.ngroups

for idx, ((event_id, article_id), group_df) in enumerate(grouped, 1):

    # Skip successfully completed articles
    if not article_results_df.empty:
        completed = (
            (article_results_df["event_id"] == event_id) &
            (article_results_df["article_id"] == article_id) &
            (article_results_df["status"] == "OK")
        )

        if completed.any():
            print(f"Skipping completed article {idx}/{total_articles}")
            continue

    print(f"\nEvaluating article {idx}/{total_articles}")
    print("Event:", event_id, "Article:", article_id)

    start = time.time()

    try:
        article_result, pair_records = evaluate_article(
            event_id,
            article_id,
            group_df
        )

        # Replace any previous failed attempt
        if not article_results_df.empty:
            mask = (
                (article_results_df["event_id"] == event_id) &
                (article_results_df["article_id"] == article_id)
            )
            article_results_df = article_results_df.loc[~mask]

        article_results_df = pd.concat(
            [article_results_df, pd.DataFrame([article_result])],
            ignore_index=True
        )

        if pair_records:
            new_pairs = pd.DataFrame(pair_records)

            if not all_pairs_df.empty:
                mask = (
                    (all_pairs_df["event_id"] == event_id) &
                    (all_pairs_df["article_id"] == article_id)
                )
                all_pairs_df = all_pairs_df.loc[~mask]

            all_pairs_df = pd.concat(
                [all_pairs_df, new_pairs],
                ignore_index=True
            )

        # Save progress after every article
        article_results_df.to_csv(ARTICLE_FILE, index=False)
        all_pairs_df.to_csv(PAIRS_FILE, index=False)

        print("Claims:", article_result["total_claims"])
        print("Candidate pairs:", article_result["candidate_pairs"])
        print("Redundant copies:", article_result["redundant_copies"])
        print(f"Redundancy: {article_result['redundancy_rate']:.2%}")

    except Exception as e:
        print("FAILED:", str(e))

        error_result = {
            "event_id": event_id,
            "article_id": article_id,
            "total_claims": len(group_df),
            "candidate_pairs": None,
            "redundant_pairs": None,
            "duplicate_groups": None,
            "redundant_copies": None,
            "redundancy_rate": None,
            "status": f"ERROR: {str(e)[:200]}"
        }

        if not article_results_df.empty:
            mask = (
                (article_results_df["event_id"] == event_id) &
                (article_results_df["article_id"] == article_id)
            )
            article_results_df = article_results_df.loc[~mask]

        article_results_df = pd.concat(
            [article_results_df, pd.DataFrame([error_result])],
            ignore_index=True
        )

        article_results_df.to_csv(ARTICLE_FILE, index=False)

    print("Elapsed seconds:", round(time.time() - start, 1))

print("\nEvaluation loop finished!")
print(article_results_df["status"].value_counts())


Evaluating article 1/30
Event: 6 Article: 6.1


Llama.generate: 238 prefix-match hit, remaining 24 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =     247.24 ms /    24 tokens (   10.30 ms per token,    97.07 tokens per second)
llama_perf_context_print:        eval time =    1976.32 ms /    68 runs   (   29.06 ms per token,    34.41 tokens per second)
llama_perf_context_print:       total time =    4719.83 ms /    92 tokens
llama_perf_context_print:    graphs reused =         67
Llama.generate: 238 prefix-match hit, remaining 20 prompt tokens to eval
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =     227.83 ms /    20 tokens (   11.39 ms per token,    87.79 tokens per second)
llama_perf_context_print:        eval time =    1481.84 ms /    51 runs   (   29.06 ms per token,    34.42 tokens per second)
llama_perf_context_print:  

Claims: 22
Candidate pairs: 21
Redundant copies: 2
Redundancy: 9.09%
Elapsed seconds: 99.3

Evaluating article 2/30
Event: 6 Article: 6.4


Llama.generate: 203 prefix-match hit, remaining 41 prompt tokens to eval
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =     243.14 ms /    41 tokens (    5.93 ms per token,   168.63 tokens per second)
llama_perf_context_print:        eval time =    1407.41 ms /    49 runs   (   28.72 ms per token,    34.82 tokens per second)
llama_perf_context_print:       total time =    3431.43 ms /    90 tokens
llama_perf_context_print:    graphs reused =         47
Llama.generate: 203 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =     229.50 ms /    32 tokens (    7.17 ms per token,   139.43 tokens per second)
llama_perf_context_print:        eval time =    1683.98 ms /    59 runs   (   28.54 ms per token,    35.04 tokens per second)
llama_perf_context_print:       total time =    4063.73 ms /    91 tokens
llama_perf_context_print:   

Claims: 22
Candidate pairs: 2
Redundant copies: 1
Redundancy: 4.55%
Elapsed seconds: 11.7

Evaluating article 3/30
Event: 6 Article: 6.5


...

Claims: 71
Candidate pairs: 13
Redundant copies: 1
Redundancy: 1.41%
Elapsed seconds: 53.3

Evaluating article 29/30
Event: 13 Article: 13.6


Llama.generate: 203 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =     230.08 ms /    32 tokens (    7.19 ms per token,   139.08 tokens per second)
llama_perf_context_print:        eval time =    1799.95 ms /    63 runs   (   28.57 ms per token,    35.00 tokens per second)
llama_perf_context_print:       total time =    4379.87 ms /    95 tokens
llama_perf_context_print:    graphs reused =         61
Llama.generate: 203 prefix-match hit, remaining 33 prompt tokens to eval
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =     229.95 ms /    33 tokens (    6.97 ms per token,   143.51 tokens per second)
llama_perf_context_print:        eval time =    1627.21 ms /    57 runs   (   28.55 ms per token,    35.03 tokens per second)
llama_perf_context_print:       total time =    3978.38 ms /    90 tokens
llama_perf_context_print:   

Claims: 33
Candidate pairs: 5
Redundant copies: 0
Redundancy: 0.00%
Elapsed seconds: 21.6

Evaluating article 30/30
Event: 13 Article: 13.5


Llama.generate: 203 prefix-match hit, remaining 33 prompt tokens to eval
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =     229.79 ms /    33 tokens (    6.96 ms per token,   143.61 tokens per second)
llama_perf_context_print:        eval time =    1395.64 ms /    49 runs   (   28.48 ms per token,    35.11 tokens per second)
llama_perf_context_print:       total time =    3447.58 ms /    82 tokens
llama_perf_context_print:    graphs reused =         47
Llama.generate: 227 prefix-match hit, remaining 8 prompt tokens to eval
llama_perf_context_print:        load time =    1551.77 ms
llama_perf_context_print: prompt eval time =      73.56 ms /     8 tokens (    9.20 ms per token,   108.75 tokens per second)
llama_perf_context_print:        eval time =    1598.19 ms /    56 runs   (   28.54 ms per token,    35.04 tokens per second)
llama_perf_context_print:       total time =    3758.00 ms /    64 tokens
llama_perf_context_print:    

Claims: 26
Candidate pairs: 2
Redundant copies: 1
Redundancy: 3.85%
Elapsed seconds: 7.7

Evaluation loop finished!
status
OK    30
Name: count, dtype: int64


In [29]:
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(
    "/project2/xiangren_1987/group_22/redundancy_results"
)

ARTICLE_FILE = RESULTS_DIR / "redundancy_by_article.csv"
SUMMARY_FILE = RESULTS_DIR / "redundancy_summary.csv"

# Load saved results
article_results_df = pd.read_csv(ARTICLE_FILE)

# Keep only successfully evaluated articles
successful = article_results_df[
    article_results_df["status"] == "OK"
].copy()

print("Successful articles:", len(successful))
print("Failed articles:", len(article_results_df) - len(successful))

Successful articles: 30
Failed articles: 0


In [30]:
total_articles = 30
THRESHOLD = 0.70

total_claims = int(successful["total_claims"].sum())
total_redundant = int(successful["redundant_copies"].sum())

overall_redundancy = (
    total_redundant / total_claims
    if total_claims > 0 else float("nan")
)

mean_article_redundancy = successful["redundancy_rate"].mean()

print("FENICE REDUNDANCY RESULTS")
print("-" * 40)
print("Articles evaluated:", len(successful))
print("Total claims:", total_claims)
print("Redundant copies:", total_redundant)
print(f"Overall redundancy rate: {overall_redundancy:.2%}")
print(f"Mean article redundancy: {mean_article_redundancy:.2%}")

FENICE REDUNDANCY RESULTS
----------------------------------------
Articles evaluated: 30
Total claims: 1014
Redundant copies: 23
Overall redundancy rate: 2.27%
Mean article redundancy: 3.13%


In [31]:
summary_df = pd.DataFrame([{
    "extractor": "FENICE T5-base",
    "redundancy_evaluator": "SBERT + Qwen3-8B",
    "sbert_model": "all-mpnet-base-v2",
    "sbert_threshold": THRESHOLD,
    "articles_evaluated": len(successful),
    "total_articles": total_articles,
    "claims_evaluated": total_claims,
    "redundant_copies": total_redundant,
    "overall_redundancy_rate": overall_redundancy,
    "mean_article_redundancy": mean_article_redundancy
}])

summary_df.to_csv(SUMMARY_FILE, index=False)

display(summary_df)

print("Summary saved successfully!")

,extractor,redundancy_evaluator,sbert_model,sbert_threshold,articles_evaluated,total_articles,claims_evaluated,redundant_copies,overall_redundancy_rate,mean_article_redundancy
0,FENICE T5-base,SBERT + Qwen3-8B,all-mpnet-base-v2,0.7,30,30,1014,23,0.022682,0.031312


Summary saved successfully!
